# Triagem de cinco classes: candidatos C0 a C4

Protocolo em `triagem_README.md` (seção 1, com as revisões R1 a R3 na 1.11).
Todas as métricas usam os logits fora da dobra gravados na pasta de trabalho.
As saídas mostram só contagens, pseudônimos e números.

## 1. Dados

In [1]:
# Bloco código 1
import time
import warnings
# Aviso do tqdm sobre barras gráficas (ipywidgets ausente); inofensivo, mas vem com
# o caminho local do arquivo, que o teste de saídas do notebook recusa
warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import pandas as pd

from meter_reader.config import get_work_dir
from meter_reader.contracts import PRODUCT_CLASSES
from training.build_manifest import file_sha256
from training.group_dataset import DEV_LABELS, check_group_sizes, load_group_photos
from training.group_labels import TRAINING_GROUP_CLASS
from training.triage_eval import BROAD_CLASS, OOF_FOLDER, load_oof, save_oof
from training.triage_features import cache_path, load_or_extract

WORK_DIR = get_work_dir()
CACHE_DIR = WORK_DIR / "caracteristicas"
LABELS_SHA = DEV_LABELS[1]

photos = load_group_photos()  # confere o SHA-256 dos rótulos (protocolo 1.1)
codes = [p.pseudonym for p in photos]
print("Fotos rotuladas:", len(photos), "| pareadas:", sum(p.paired for p in photos))
print("Grupos de treino:", check_group_sizes(photos))
for name in ("mobilenet_v3_large_224", "dinov2_224"):
    print(f"Cache {name}: SHA-256 {file_sha256(cache_path(CACHE_DIR, name, 5))}")

Fotos rotuladas: 9301 | pareadas: 8637
Grupos de treino: {'dig1_quadrado_branco': 6372, 'dig2_redondo_branco': 208, 'dig3_vidro_redondo': 369, 'dig4_verde_arduino': 76, 'cic_disco': 616, 'cic_duas_leituras': 173, 'cic_uma_leitura': 658, 'indeterminado': 255, 'multiplos': 72, 'out_acesso': 307, 'out_sem_acesso': 105, 'out_sem_contexto': 90}
Cache mobilenet_v3_large_224: SHA-256 c44b4f96623d3574482f351141125aa98e723d9b0dd81565d17e68b78a35a8d1
Cache dinov2_224: SHA-256 30e002f5feacf1ba58d17fdfaaf9247b7b6eb04cd379daa566b303a93ab0b28f


## 2. C0: MobileNetV3-Large congelada e sonda linear

Três execuções: W-b com rótulo fino (padrão), W-d com rótulo fino (ablação de
pesos, 1.4) e W-b com rótulo amplo (ablação fino × amplo, 1.5). Uma execução
por dobra (revisão R1). Abaixo, a evidência da R1 nesta base.

In [2]:
# Bloco código 2
from sklearn.linear_model import LogisticRegression

c0_feats = load_or_extract([p.path for p in photos], codes, "mobilenet_v3_large_224", 5, CACHE_DIR)
rng = np.random.default_rng(0)
sub = rng.choice(len(photos), size=1500, replace=False)  # subamostra: só a evidência
x_sub = c0_feats[sub, 0]
x_sub = (x_sub - x_sub.mean(axis=0)) / (x_sub.std(axis=0) + 1e-6)
y_sub = np.array([photos[i].target for i in sub])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # a subamostra pode não convergir; aqui só importa a igualdade
    a = LogisticRegression(max_iter=2000, random_state=0).fit(x_sub, y_sub).coef_
    b = LogisticRegression(max_iter=2000, random_state=99).fit(x_sub, y_sub).coef_
print("Coeficientes idênticos com random_state 0 e 99:", np.array_equal(a, b))

Coeficientes idênticos com random_state 0 e 99: True


In [3]:
# Bloco código 3
from training.linear_probe import cross_validate_probe

EVAL_CODES = None  # ordem das pareadas avaliadas, igual em todas as execuções


def run_probe(name, feats, targets, output_class, weighting, config):
    """Treina e grava os logits fora da dobra; se já gravados, só lê."""
    global EVAL_CODES
    if (WORK_DIR / OOF_FOLDER / f"{name}.npz").exists():
        data = np.load(WORK_DIR / OOF_FOLDER / f"{name}.npz")
        EVAL_CODES = EVAL_CODES or list(data["codes"])
        print(f"{name}: lido do arquivo gravado")
        return load_oof(WORK_DIR, name, EVAL_CODES)
    start = time.perf_counter()
    logits, index, folds = cross_validate_probe(feats, photos, targets, output_class, weighting)
    eval_codes = [codes[i] for i in index]
    EVAL_CODES = EVAL_CODES or eval_codes
    assert eval_codes == EVAL_CODES, "Ordem das fotos avaliadas diferente entre execuções."
    save_oof(WORK_DIR, name, eval_codes, [photos[i].lot for i in index],
             logits[None], {**config, "ponderacao": weighting,
                            "dobras": [vars(f) for f in folds]}, LABELS_SHA)
    minutes = (time.perf_counter() - start) / 60
    print(f"{name}: {minutes:.1f} min; iterações por dobra {[f.n_iter for f in folds]}")
    return logits


fine = np.array([p.target for p in photos])
broad = np.array([p.product_class for p in photos])
base = {"candidato": "C0", "extrator": "mobilenet_v3_large_224", "vistas": 5}

oof = {
    "C0 W-b fino": run_probe("c0_wb_fino", c0_feats, fine, TRAINING_GROUP_CLASS, "W-b",
                             {**base, "rotulo": "fino"}),
    "C0 W-d fino": run_probe("c0_wd_fino", c0_feats, fine, TRAINING_GROUP_CLASS, "W-d",
                             {**base, "rotulo": "fino"}),
    "C0 W-b amplo": run_probe("c0_wb_amplo", c0_feats, broad, BROAD_CLASS, "W-b",
                              {**base, "rotulo": "amplo"}),
}

c0_wb_fino: lido do arquivo gravado
c0_wd_fino: lido do arquivo gravado
c0_wb_amplo: lido do arquivo gravado


### Avaliação do C0 e decisão das ablações

Cada execução: temperatura aninhada por lote (1.6), escore de rejeição, limiar
da meta de perda ≤ 0,5% pelo limite superior de Clopper-Pearson e cobertura da
rejeição nesse limiar (1.7). As ablações comparam a cobertura de cada execução
no seu limiar, por bootstrap pareado (2.000 reamostragens, semente 0).

In [4]:
# Bloco código 4
from training.triage_eval import (DETERMINED, calibrate_nested, choose_threshold,
                                  loss_coverage_curve, paired_coverage_diff, rejection_score)

by_code = {p.pseudonym: p for p in photos}
eval_photos = [by_code[c] for c in EVAL_CODES]
labels = np.array([p.product_class for p in eval_photos])
lots = np.array([p.lot for p in eval_photos])
print("Avaliadas:", len(labels), "| determinados:", int(np.isin(labels, DETERMINED).sum()),
      "| rejeitáveis:", int((~np.isin(labels, DETERMINED)).sum()))

OUTPUT_CLASS = {"C0 W-b fino": TRAINING_GROUP_CLASS, "C0 W-d fino": TRAINING_GROUP_CLASS,
                "C0 W-b amplo": BROAD_CLASS}
results, scores = {}, {}


def evaluate(name, logits, output_class):
    """Calibração aninhada, curva, limiar da meta e cobertura."""
    cal = calibrate_nested(logits, labels, lots, output_class)
    score = rejection_score(cal.probs)
    best = choose_threshold(loss_coverage_curve(score, labels))
    scores[name] = (score, None if best is None else best["threshold"])
    return {
        "T por lote": " / ".join(f"{t:.2f}" for t in cal.per_lot.values()),
        "T final": round(cal.final, 3),
        "limiar": None if best is None else round(best["threshold"], 4),
        "perdas": None if best is None else int(best["losses"]),
        "perda": None if best is None else round(best["loss"], 4),
        "limite sup. perda": None if best is None else round(best["loss_upper"], 4),
        "cobertura": 0.0 if best is None else round(best["coverage"], 4),
    }


for name, logits in oof.items():
    results[name] = evaluate(name, logits, OUTPUT_CLASS[name])
pd.DataFrame(results).T

Avaliadas: 8637 | determinados: 7873 | rejeitáveis: 764


,T por lote,T final,limiar,perdas,perda,limite sup. perda,cobertura
C0 W-b fino,4.58 / 4.48 / 4.62,4.555,0.9046,28,0.0036,0.0049,0.4581
C0 W-d fino,4.39 / 4.38 / 4.32,4.361,0.8808,28,0.0036,0.0049,0.4476
C0 W-b amplo,3.93 / 3.91 / 3.86,3.901,0.9495,28,0.0036,0.0049,0.4398


In [5]:
# Bloco código 5
def compare(a, b):
    """Cobertura(a) − cobertura(b), cada um no seu limiar, com IC 95%."""
    (sa, ta), (sb, tb) = scores[a], scores[b]
    point, low, high = paired_coverage_diff(sa, ta, sb, tb, labels)
    return {"diferença": round(point, 4), "IC 95%": f"[{low:+.4f}; {high:+.4f}]",
            "vence a ablação": low > 0}


ablations = {
    "W-d − W-b (1.4)": compare("C0 W-d fino", "C0 W-b fino"),
    "amplo − fino (1.5)": compare("C0 W-b amplo", "C0 W-b fino"),
}
pd.DataFrame(ablations).T

,diferença,IC 95%,vence a ablação
W-d − W-b (1.4),-0.0105,[-0.0242; +0.0039],False
amplo − fino (1.5),-0.0183,[-0.0427; +0.0052],False


## 3. C1: DINOv2 congelado e sonda linear

Mesma sonda e mesma configuração vencedora das ablações do C0 (W-b, rótulo
fino), sobre o cache novo das vistas (revisão R2).

In [6]:
# Bloco código 6
c1_feats = load_or_extract([p.path for p in photos], codes, "dinov2_224", 5, CACHE_DIR)
oof["C1"] = run_probe("c1_wb_fino", c1_feats, fine, TRAINING_GROUP_CLASS, "W-b",
                      {"candidato": "C1", "extrator": "dinov2_224", "vistas": 5,
                       "rotulo": "fino"})
OUTPUT_CLASS["C1"] = TRAINING_GROUP_CLASS
results["C1"] = evaluate("C1", oof["C1"], TRAINING_GROUP_CLASS)
pd.DataFrame(results).T

c1_wb_fino: lido do arquivo gravado


,T por lote,T final,limiar,perdas,perda,limite sup. perda,cobertura
C0 W-b fino,4.58 / 4.48 / 4.62,4.555,0.9046,28,0.0036,0.0049,0.4581
C0 W-d fino,4.39 / 4.38 / 4.32,4.361,0.8808,28,0.0036,0.0049,0.4476
C0 W-b amplo,3.93 / 3.91 / 3.86,3.901,0.9495,28,0.0036,0.0049,0.4398
C1,2.50 / 2.49 / 2.58,2.525,0.9167,28,0.0036,0.0049,0.5563


## 4. C2: MobileNetV3-Large com ajuste fino

Rede inteira em resolução nativa, 3 sementes por dobra, combinadas pela média
dos logits (E31). Treinado pelo comando `training.run_finetune` do produto.
A ablação W-d × W-b decide os pesos da família ajustada (C2, C3 e C4; seção 1.4).

In [7]:
# Bloco código 7
import json

for name in ("c2_wb_fino", "c2_wd_fino"):
    meta = json.loads((WORK_DIR / OOF_FOLDER / f"{name}.json").read_text(encoding="utf-8"))
    epochs = list(meta["configuracao"]["epocas_escolhidas"].values())
    print(f"{name}: commit {meta['commit_produto'][:7]}, sementes {meta['sementes']}, "
          f"épocas escolhidas {epochs} (mediana {int(np.median(epochs))})")

oof["C2 W-b"] = load_oof(WORK_DIR, "c2_wb_fino", EVAL_CODES)
oof["C2 W-d"] = load_oof(WORK_DIR, "c2_wd_fino", EVAL_CODES)
for name in ("C2 W-b", "C2 W-d"):
    OUTPUT_CLASS[name] = TRAINING_GROUP_CLASS
    results[name] = evaluate(name, oof[name], TRAINING_GROUP_CLASS)

ablations["C2: W-d − W-b (1.4)"] = compare("C2 W-d", "C2 W-b")
display(pd.DataFrame(results).T)
pd.DataFrame(ablations).T

c2_wb_fino: commit abcb7f8, sementes 3, épocas escolhidas [3, 2, 4, 10, 4, 3, 2, 2, 3] (mediana 3)
c2_wd_fino: commit abcb7f8, sementes 3, épocas escolhidas [4, 3, 3, 5, 3, 4, 7, 3, 2] (mediana 3)


,T por lote,T final,limiar,perdas,perda,limite sup. perda,cobertura
C0 W-b fino,4.58 / 4.48 / 4.62,4.555,0.9046,28,0.0036,0.0049,0.4581
C0 W-d fino,4.39 / 4.38 / 4.32,4.361,0.8808,28,0.0036,0.0049,0.4476
C0 W-b amplo,3.93 / 3.91 / 3.86,3.901,0.9495,28,0.0036,0.0049,0.4398
C1,2.50 / 2.49 / 2.58,2.525,0.9167,28,0.0036,0.0049,0.5563
C2 W-b,1.01 / 1.18 / 1.14,1.109,0.8903,28,0.0036,0.0049,0.5982
C2 W-d,1.13 / 1.27 / 1.25,1.217,0.7935,28,0.0036,0.0049,0.5065


,diferença,IC 95%,vence a ablação
W-d − W-b (1.4),-0.0105,[-0.0242; +0.0039],False
amplo − fino (1.5),-0.0183,[-0.0427; +0.0052],False
C2: W-d − W-b (1.4),-0.0916,[-0.1192; -0.0644],False


## 5. C3: DINOv2 com ajuste fino

Rede inteira a 224 px (corte em múltiplo de 14), W-b (decidido no C2 para a
família ajustada), 3 sementes por dobra. Treinado pelo `training.run_finetune`.

In [8]:
# Bloco código 8
meta = json.loads((WORK_DIR / OOF_FOLDER / "c3_wb_fino.json").read_text(encoding="utf-8"))
epochs = list(meta["configuracao"]["epocas_escolhidas"].values())
print(f"c3_wb_fino: commit {meta['commit_produto'][:7]}, sementes {meta['sementes']}, "
      f"épocas escolhidas {epochs} (mediana {int(np.median(epochs))})")

oof["C3"] = load_oof(WORK_DIR, "c3_wb_fino", EVAL_CODES)
OUTPUT_CLASS["C3"] = TRAINING_GROUP_CLASS
results["C3"] = evaluate("C3", oof["C3"], TRAINING_GROUP_CLASS)
pd.DataFrame(results).T

c3_wb_fino: commit abcb7f8, sementes 3, épocas escolhidas [2, 7, 3, 8, 5, 4, 3, 4, 3] (mediana 4)


,T por lote,T final,limiar,perdas,perda,limite sup. perda,cobertura
C0 W-b fino,4.58 / 4.48 / 4.62,4.555,0.9046,28,0.0036,0.0049,0.4581
C0 W-d fino,4.39 / 4.38 / 4.32,4.361,0.8808,28,0.0036,0.0049,0.4476
C0 W-b amplo,3.93 / 3.91 / 3.86,3.901,0.9495,28,0.0036,0.0049,0.4398
C1,2.50 / 2.49 / 2.58,2.525,0.9167,28,0.0036,0.0049,0.5563
C2 W-b,1.01 / 1.18 / 1.14,1.109,0.8903,28,0.0036,0.0049,0.5982
C2 W-d,1.13 / 1.27 / 1.25,1.217,0.7935,28,0.0036,0.0049,0.5065
C3,1.49 / 1.36 / 1.56,1.469,0.845,28,0.0036,0.0049,0.5877


## 6. Revisão R3 das discordâncias (seção 1.11)

União das 50 fotos com a menor probabilidade calibrada da classe do rótulo em
cada candidato (C0, C1, C2 e C3, nas configurações escolhidas), só pareadas.
A lista vai para a pasta de trabalho em ordem aleatória, sem rótulos, para a
revisão às cegas.

In [9]:
# Bloco código 9
REVIEWED = ["C0 W-b fino", "C1", "C2 W-b", "C3"]
K = 50
REVIEW_DIR = WORK_DIR / "revisao_r3"

selected = {}
for name in REVIEWED:
    cal = calibrate_nested(oof[name], labels, lots, OUTPUT_CLASS[name])
    p_label = cal.probs[np.arange(len(labels)), labels]  # prob. calibrada da classe do rótulo
    selected[name] = set(np.argsort(p_label, kind="stable")[:K].tolist())

union = sorted(set().union(*selected.values()))
overlap = pd.DataFrame({a: [len(selected[a] & selected[b]) for b in REVIEWED] for a in REVIEWED},
                       index=REVIEWED)
print("Fotos na união:", len(union))
print("Por classe do rótulo:", dict(pd.Series([PRODUCT_CLASSES[labels[i]] for i in union]).value_counts()))
display(overlap)

# Lista para a revisão às cegas: só pseudônimos, em ordem aleatória fixa
REVIEW_DIR.mkdir(exist_ok=True)
order = np.random.default_rng(0).permutation(union)
pd.DataFrame({"pseudonimo": [EVAL_CODES[i] for i in order]}).to_csv(
    REVIEW_DIR / "fotos_revisao.csv", index=False)
print("Lista gravada na pasta de trabalho:", len(order), "pseudônimos")

Fotos na união: 150
Por classe do rótulo: {'indeterminado': np.int64(61), 'ciclometrico': np.int64(29), 'digital': np.int64(26), 'outros': np.int64(21), 'multiplos': np.int64(13)}


,C0 W-b fino,C1,C2 W-b,C3
C0 W-b fino,50,10,12,13
C1,10,50,5,11
C2 W-b,12,5,50,18
C3,13,11,18,50


Lista gravada na pasta de trabalho: 150 pseudônimos


### Resultado da revisão R3

Correção de grupo: o grupo final difere do atual (muda o rótulo de treino).
Correção só de condição: não altera a triagem. Gatilho de retreino (1.11, R3):
mais de 1% das 9.301 fotos com correção de grupo, ou mais de 10% de alguma
classe de rejeição com fotos entrando ou saindo dela (leitura mais rigorosa
da regra, fixada antes de ver as contagens).

In [10]:
# Bloco código 10
from training.group_labels import product_class as group_class

review = pd.read_csv(REVIEW_DIR / "revisao_r3.csv", dtype=str)
assert len(review) == len(union) and set(review["pseudonimo"]) == {EVAL_CODES[i] for i in union}

for col in ("cego", "atual", "final"):
    review[f"classe_{col}"] = review[f"grupo_{col}"].map(group_class)

n = len(review)
blind_group = (review["grupo_cego"] == review["grupo_atual"]).mean()
blind_class = (review["classe_cego"] == review["classe_atual"]).mean()
group_fix = review["grupo_final"] != review["grupo_atual"]
cond_only = ~group_fix & (review["condicao_final"] != review["condicao_atual"])
print(f"Fotos revisadas: {n}")
print(f"Escolha às cegas igual ao rótulo: grupo {blind_group:.1%}, classe do produto {blind_class:.1%}")
print(f"Correções de grupo: {group_fix.sum()} | só de condição: {cond_only.sum()}")

# Transições de classe do produto entre o rótulo atual e o final
moved = review[review["classe_final"] != review["classe_atual"]]
display(pd.crosstab(moved["classe_atual"], moved["classe_final"],
                    rownames=["atual"], colnames=["final"]))

# Gatilho de retreino
dev_counts = pd.Series([PRODUCT_CLASSES[p.product_class] for p in photos]).value_counts()
trigger = {"correções de grupo / 9.301": group_fix.sum() / len(photos)}
for c in ("indeterminado", "multiplos", "outros"):
    touched = ((moved["classe_atual"] == c) | (moved["classe_final"] == c)).sum()
    trigger[f"{c}: entram ou saem / {dev_counts[c]}"] = touched / dev_counts[c]
limits = [0.01, 0.10, 0.10, 0.10]
table = pd.DataFrame({"fração": trigger.values(), "limite": limits}, index=trigger.keys())
table["dispara"] = table["fração"] > table["limite"]
display(table)
print("Retreino necessário:", bool(table["dispara"].any()))

Fotos revisadas: 150
Escolha às cegas igual ao rótulo: grupo 53.3%, classe do produto 58.0%
Correções de grupo: 51 | só de condição: 4


final,ciclometrico,digital,indeterminado,multiplos,outros
atual,,,,,
ciclometrico,0,4,7,1,0
digital,6,0,6,2,0
indeterminado,2,7,0,0,1
multiplos,0,0,1,0,0
outros,0,4,6,1,0


,fração,limite,dispara
correções de grupo / 9.301,0.005483,0.01,False
indeterminado: entram ou saem / 255,0.117647,0.10,True
multiplos: entram ou saem / 72,0.069444,0.10,False
outros: entram ou saem / 502,0.023904,0.10,False


Retreino necessário: True


## 7. Candidatos com os rótulos corrigidos (R3)

A revisão disparou o gatilho de retreino (indeterminado: 30 de 255 fotos
entrando ou saindo, 11,8% > 10%). Todos os candidatos e ablações foram
retreinados com os rótulos R3 (opção B), inclusive o C2 W-d. As fotos avaliadas
são as mesmas; só os rótulos mudam. A regra Δ2 é aplicada sobre estes resultados.

In [11]:
# Bloco código 11
from training.group_dataset import DEV_LABELS_R3

photos_r3 = load_group_photos(DEV_LABELS_R3)
assert [p.pseudonym for p in photos_r3] == codes, "Fotos ou ordem diferentes entre as versões."
fine_r3 = np.array([p.target for p in photos_r3])
broad_r3 = np.array([p.product_class for p in photos_r3])


def run_probe_r3(name, feats, targets, output_class, weighting, config):
    """Como run_probe, com os rótulos R3; grava uma vez e depois só lê."""
    if (WORK_DIR / OOF_FOLDER / f"{name}.npz").exists():
        print(f"{name}: lido do arquivo gravado")
        return load_oof(WORK_DIR, name, EVAL_CODES)
    logits, index, folds = cross_validate_probe(feats, photos_r3, targets, output_class, weighting)
    assert [codes[i] for i in index] == EVAL_CODES
    save_oof(WORK_DIR, name, EVAL_CODES, [photos_r3[i].lot for i in index], logits[None],
             {**config, "ponderacao": weighting, "versao_rotulos": "r3",
              "dobras": [vars(f) for f in folds]}, DEV_LABELS_R3[1])
    print(f"{name}: iterações por dobra {[f.n_iter for f in folds]}")
    return logits


base0 = {"candidato": "C0", "extrator": "mobilenet_v3_large_224", "vistas": 5}
oof_r3 = {
    "C0 W-b fino": run_probe_r3("c0_wb_fino_r3", c0_feats, fine_r3, TRAINING_GROUP_CLASS, "W-b",
                                {**base0, "rotulo": "fino"}),
    "C0 W-d fino": run_probe_r3("c0_wd_fino_r3", c0_feats, fine_r3, TRAINING_GROUP_CLASS, "W-d",
                                {**base0, "rotulo": "fino"}),
    "C0 W-b amplo": run_probe_r3("c0_wb_amplo_r3", c0_feats, broad_r3, BROAD_CLASS, "W-b",
                                 {**base0, "rotulo": "amplo"}),
    "C1": run_probe_r3("c1_wb_fino_r3", c1_feats, fine_r3, TRAINING_GROUP_CLASS, "W-b",
                       {"candidato": "C1", "extrator": "dinov2_224", "vistas": 5, "rotulo": "fino"}),
    "C2 W-b": load_oof(WORK_DIR, "c2_wb_fino_r3", EVAL_CODES),
    "C2 W-d": load_oof(WORK_DIR, "c2_wd_fino_r3", EVAL_CODES),
    "C3": load_oof(WORK_DIR, "c3_wb_fino_r3", EVAL_CODES),
}

# Avaliação com os rótulos R3: as funções evaluate e compare usam estes globais
results_v1, ablations_v1 = results, ablations   # registro do antes
by_code_r3 = {p.pseudonym: p for p in photos_r3}
labels = np.array([by_code_r3[c].product_class for c in EVAL_CODES])
scores, results = {}, {}
for name, logits in oof_r3.items():
    results[name] = evaluate(name, logits, OUTPUT_CLASS[name])
ablations = {
    "C0: W-d − W-b (1.4)": compare("C0 W-d fino", "C0 W-b fino"),
    "C0: amplo − fino (1.5)": compare("C0 W-b amplo", "C0 W-b fino"),
    "C2: W-d − W-b (1.4)": compare("C2 W-d", "C2 W-b"),
}
print("Determinados:", int(np.isin(labels, DETERMINED).sum()),
      "| rejeitáveis:", int((~np.isin(labels, DETERMINED)).sum()))
display(pd.DataFrame(results).T)
pd.DataFrame(ablations).T

c0_wb_fino_r3: lido do arquivo gravado
c0_wd_fino_r3: lido do arquivo gravado
c0_wb_amplo_r3: lido do arquivo gravado
c1_wb_fino_r3: lido do arquivo gravado
Determinados: 7870 | rejeitáveis: 767


,T por lote,T final,limiar,perdas,perda,limite sup. perda,cobertura
C0 W-b fino,4.47 / 4.43 / 4.43,4.444,0.9161,27,0.0034,0.0047,0.4733
C0 W-d fino,4.36 / 4.32 / 4.24,4.305,0.8626,28,0.0036,0.0049,0.4863
C0 W-b amplo,3.86 / 3.97 / 3.82,3.881,0.9413,28,0.0036,0.0049,0.4772
C1,2.49 / 2.56 / 2.55,2.534,0.9161,28,0.0036,0.0049,0.5919
C2 W-b,1.12 / 1.19 / 0.99,1.098,0.8598,28,0.0036,0.0049,0.6076
C2 W-d,1.28 / 1.30 / 1.20,1.262,0.7991,27,0.0034,0.0047,0.5711
C3,1.47 / 1.48 / 1.43,1.458,0.8204,28,0.0036,0.0049,0.678


,diferença,IC 95%,vence a ablação
C0: W-d − W-b (1.4),0.013,[+0.0013; +0.0254],True
C0: amplo − fino (1.5),0.0039,[-0.0210; +0.0289],False
C2: W-d − W-b (1.4),-0.0365,[-0.0636; -0.0102],False


In [12]:
# Bloco código 12
# 1) C1 com W-d (a ablação do C0 venceu: W-d vale para a família congelada, seção 1.4)
oof_r3["C1 W-d"] = run_probe_r3("c1_wd_fino_r3", c1_feats, fine_r3, TRAINING_GROUP_CLASS, "W-d",
                                {"candidato": "C1", "extrator": "dinov2_224", "vistas": 5,
                                 "rotulo": "fino"})
OUTPUT_CLASS["C1 W-d"] = TRAINING_GROUP_CLASS
results["C1 W-d"] = evaluate("C1 W-d", oof_r3["C1 W-d"], TRAINING_GROUP_CLASS)


def coverage_at_target(logits, output_class=TRAINING_GROUP_CLASS):
    """Cobertura no limiar da meta, com os rótulos R3 (sem gravar nada)."""
    cal = calibrate_nested(logits, labels, lots, output_class)
    best = choose_threshold(loss_coverage_curve(rejection_score(cal.probs), labels))
    return 0.0 if best is None else float(best["coverage"])


# 2) Diagnóstico: logits da v1 avaliados com os rótulos R3 (isola o efeito da avaliação)
diag = {}
for cand, v1_name, r3_name in (("C2 W-b", "c2_wb_fino", "c2_wb_fino_r3"),
                               ("C3", "c3_wb_fino", "c3_wb_fino_r3")):
    row = {"v1 treinado, rótulos R3": coverage_at_target(oof[cand]),
           "R3 treinado, rótulos R3": coverage_at_target(oof_r3[cand])}
    # 3) Cobertura de cada semente separada (ruído de treino), nas duas versões
    for tag, name in (("v1", v1_name), ("R3", r3_name)):
        per_seed = np.load(WORK_DIR / OOF_FOLDER / f"{name}.npz")["logits"]
        seeds = [coverage_at_target(per_seed[s]) for s in range(per_seed.shape[0])]
        row[f"sementes {tag}"] = " / ".join(f"{c:.3f}" for c in seeds)
    diag[cand] = row
display(pd.DataFrame(results).T)
pd.DataFrame(diag).T

c1_wd_fino_r3: lido do arquivo gravado


,T por lote,T final,limiar,perdas,perda,limite sup. perda,cobertura
C0 W-b fino,4.47 / 4.43 / 4.43,4.444,0.9161,27,0.0034,0.0047,0.4733
C0 W-d fino,4.36 / 4.32 / 4.24,4.305,0.8626,28,0.0036,0.0049,0.4863
C0 W-b amplo,3.86 / 3.97 / 3.82,3.881,0.9413,28,0.0036,0.0049,0.4772
C1,2.49 / 2.56 / 2.55,2.534,0.9161,28,0.0036,0.0049,0.5919
C2 W-b,1.12 / 1.19 / 0.99,1.098,0.8598,28,0.0036,0.0049,0.6076
C2 W-d,1.28 / 1.30 / 1.20,1.262,0.7991,27,0.0034,0.0047,0.5711
C3,1.47 / 1.48 / 1.43,1.458,0.8204,28,0.0036,0.0049,0.678
C1 W-d,2.87 / 2.95 / 2.97,2.932,0.8842,28,0.0036,0.0049,0.5854


,"v1 treinado, rótulos R3","R3 treinado, rótulos R3",sementes v1,sementes R3
C2 W-b,0.640156,0.607562,0.512 / 0.515 / 0.592,0.489 / 0.508 / 0.592
C3,0.646675,0.677966,0.546 / 0.611 / 0.617,0.645 / 0.508 / 0.584


### Regra de escolha Δ2 (seção 1.7), com os rótulos R3

Configurações pelas ablações R3: C0 e C1 com W-d (família congelada); C2 e C3
com W-b (família ajustada); rótulo fino. Ordem de simplicidade: C0, C2, C4,
C1, C3. Vence o primeiro cujo limite inferior do IC 95% da diferença de
cobertura em relação ao melhor seja maior que −0,05. Diagnóstico relatado,
fora da regra: a cobertura de uma semente isolada varia de 10 a 14 pontos no
C2 e no C3; o bootstrap da regra não mede essa variação de treino.

In [13]:
# Bloco código 13
CHOSEN = {"C0": "C0 W-d fino", "C2": "C2 W-b", "C1": "C1 W-d", "C3": "C3"}
ORDER = ["C0", "C2", "C4", "C1", "C3"]   # seção 1.3 (opção S-a)
MARGIN = -0.05

best = max(CHOSEN, key=lambda c: results[CHOSEN[c]]["cobertura"])
rule = {}
for cand in ORDER:
    if cand not in CHOSEN:
        rule[cand] = {"situação": "não treinado"}
        continue
    point, low, high = paired_coverage_diff(*scores[CHOSEN[cand]], *scores[CHOSEN[best]], labels)
    rule[cand] = {"cobertura": results[CHOSEN[cand]]["cobertura"],
                  f"diferença para {best}": round(point, 4),
                  "IC 95%": f"[{low:+.4f}; {high:+.4f}]",
                  "não inferior": low > MARGIN}
display(pd.DataFrame(rule).T)

winner = next((c for c in ORDER if c in CHOSEN and rule[c]["não inferior"]), None)
c4_needed = not (rule["C0"]["não inferior"] or rule["C2"]["não inferior"])
print("Melhor cobertura:", best)
print("C4 exigido pela regra:", c4_needed)
print("Vencedor pela regra (sem o C4):", winner)

,cobertura,diferença para C3,IC 95%,não inferior,situação
C0,0.4863,-0.1917,[-0.2244; -0.1571],False,NaN
C2,0.6076,-0.0704,[-0.0990; -0.0424],False,NaN
C4,NaN,NaN,NaN,NaN,não treinado
C1,0.5854,-0.0926,[-0.1242; -0.0598],False,NaN
C3,0.678,0.0,[+0.0000; +0.0000],True,NaN


Melhor cobertura: C3
C4 exigido pela regra: True
Vencedor pela regra (sem o C4): C3


### C4 (destilação do C3 para o C2) e regra Δ2 final

C4 treinado porque nem o C0 nem o C2 foram não inferiores ao C3 (seção 1.3).
Professor: C3 retreinado em cada dobra e semente; suas predições medem a
reprodutibilidade do C3.

In [14]:
# Bloco código 14
meta = json.loads((WORK_DIR / OOF_FOLDER / "c4_wb_fino_r3.json").read_text(encoding="utf-8"))
epochs = list(meta["configuracao"]["epocas_escolhidas"].values())
print(f"C4: commit {meta['commit_produto'][:7]}, épocas escolhidas {epochs} "
      f"(mediana {int(np.median(epochs))})")

oof_r3["C4"] = load_oof(WORK_DIR, "c4_wb_fino_r3", EVAL_CODES)
OUTPUT_CLASS["C4"] = TRAINING_GROUP_CLASS
results["C4"] = evaluate("C4", oof_r3["C4"], TRAINING_GROUP_CLASS)

# Reprodutibilidade do C3: professor retreinado (partes) contra o C3 avaliado
parts_dir = WORK_DIR / OOF_FOLDER / "partes" / "c4_wb_fino_r3_professor"
teacher_seeds = []
for seed in range(3):
    parts = [np.load(parts_dir / f"{lot}_s{seed}.npz") for lot in sorted(set(lots))]
    assert [c for p in parts for c in p["codes"]] == EVAL_CODES
    teacher_seeds.append(np.concatenate([p["logits"] for p in parts]))
print(f"C3 avaliado: {results['C3']['cobertura']:.3f} | C3 retreinado (professor): "
      f"{coverage_at_target(np.mean(teacher_seeds, axis=0)):.3f} | sementes do professor: "
      + " / ".join(f"{coverage_at_target(t):.3f}" for t in teacher_seeds))

# Regra Δ2 final, agora com o C4
CHOSEN["C4"] = "C4"
best = max(CHOSEN, key=lambda c: results[CHOSEN[c]]["cobertura"])
rule = {}
for cand in ORDER:
    point, low, high = paired_coverage_diff(*scores[CHOSEN[cand]], *scores[CHOSEN[best]], labels)
    rule[cand] = {"cobertura": results[CHOSEN[cand]]["cobertura"],
                  f"diferença para {best}": round(point, 4),
                  "IC 95%": f"[{low:+.4f}; {high:+.4f}]", "não inferior": low > MARGIN}
display(pd.DataFrame(rule).T)
winner = next(c for c in ORDER if rule[c]["não inferior"])
print("Melhor cobertura:", best, "| Vencedor pela regra Δ2:", winner)

C4: commit fe543c4, épocas escolhidas [15, 14, 12, 11, 9, 11, 5, 9, 8] (mediana 11)
C3 avaliado: 0.678 | C3 retreinado (professor): 0.640 | sementes do professor: 0.549 / 0.604 / 0.548


,cobertura,diferença para C3,IC 95%,não inferior
C0,0.4863,-0.1917,[-0.2244; -0.1571],False
C2,0.6076,-0.0704,[-0.0990; -0.0424],False
C4,0.6532,-0.0248,[-0.0526; +0.0038],False
C1,0.5854,-0.0926,[-0.1242; -0.0598],False
C3,0.678,0.0,[+0.0000; +0.0000],True


Melhor cobertura: C3 | Vencedor pela regra Δ2: C3


### Relatórios da seção 1.8 (desenvolvimento, rótulos R3)

C4: adotado pelo produto (decisão de produto: custo de inferência cerca de 5
vezes menor; diferença de qualidade dentro do ruído de treino medido).
C3: vencedor da regra Δ2 do estudo. Os dois são avaliados na abertura do teste,
cada um no seu papel, fixado antes da abertura (revisão R4).

In [15]:
# Bloco código 15
from training.calibration import ece
from training.triage_eval import REJECTABLE, class_log_probs, rejection_destination

REPORTED = ["C4", "C3"]
group_of = np.array([by_code_r3[c].group for c in EVAL_CODES])
cond_of = np.array([by_code_r3[c].condition for c in EVAL_CODES])
determined = np.isin(labels, DETERMINED)


def loss_and_coverage(rejected, mask):
    d, r = mask & determined, mask & ~determined
    return {"perdas": f"{int((rejected & d).sum())}/{int(d.sum())}",
            "cobertura": f"{(rejected & r).sum() / max(1, r.sum()):.3f} ({int(r.sum())})"}


by_lot, by_cond, extras, calibration = {}, {}, {}, {}
for m in REPORTED:
    cal = calibrate_nested(oof_r3[m], labels, lots, OUTPUT_CLASS[m])
    before = np.exp(class_log_probs(oof_r3[m], 1.0, OUTPUT_CLASS[m]))
    score, threshold = scores[m]
    rejected = score >= threshold
    dest = rejection_destination(cal.probs)

    for lot in sorted(set(lots)):
        by_lot[(m, lot)] = loss_and_coverage(rejected, lots == lot)
    for cond in sorted(set(cond_of)):
        by_cond[(m, cond)] = loss_and_coverage(rejected, cond_of == cond)

    hit = rejected & ~determined
    two = group_of == "cic_duas_leituras"
    extras[m] = {
        "destino trocado": f"{int((dest[hit] != labels[hit]).sum())}/{int(hit.sum())}",
        "duas leituras: argmax múltiplos": f"{int((cal.probs[two].argmax(1) == PRODUCT_CLASSES.index('multiplos')).sum())}/{int(two.sum())}",
        "duas leituras: rejeitadas": int(rejected[two].sum()),
    }
    for tag, probs in (("antes", before), ("depois", cal.probs)):
        calibration[(m, tag)] = {"P(rejeitável)": ece(probs[:, list(REJECTABLE)].sum(1), ~determined)}
        for c in REJECTABLE:
            calibration[(m, tag)][f"P({PRODUCT_CLASSES[c]})"] = ece(probs[:, c], labels == c)

    print(f"\nMatriz de confusão de {m} (linhas: rótulo; colunas: classe mais provável)")
    display(pd.crosstab(pd.Series([PRODUCT_CLASSES[i] for i in labels], name="rótulo"),
                        pd.Series([PRODUCT_CLASSES[i] for i in cal.probs.argmax(1)], name="prevista")))

display(pd.DataFrame(by_lot).T)
display(pd.DataFrame(by_cond).T)
display(pd.DataFrame(extras).T)
pd.DataFrame(calibration).T.round(4)


Matriz de confusão de C4 (linhas: rótulo; colunas: classe mais provável)


prevista,ciclometrico,digital,indeterminado,multiplos,outros
rótulo,,,,,
ciclometrico,1330,66,12,1,5
digital,22,6339,70,6,19
indeterminado,11,61,138,2,41
multiplos,4,30,9,27,2
outros,2,11,25,3,401



Matriz de confusão de C3 (linhas: rótulo; colunas: classe mais provável)


prevista,ciclometrico,digital,indeterminado,multiplos,outros
rótulo,,,,,
ciclometrico,1331,70,11,0,2
digital,27,6348,69,2,10
indeterminado,14,71,132,2,34
multiplos,3,21,8,39,1
outros,0,14,35,5,388


perdas    cobertura
C4 2026-05-20   6/2721  0.649 (245)
   2026-05-21  15/2606  0.697 (264)
   2026-05-22   7/2543  0.612 (258)
C3 2026-05-20   7/2721  0.645 (245)
   2026-05-21  13/2606  0.682 (264)
   2026-05-22   8/2543  0.705 (258)

perdas    cobertura
C4 caixa             1/318    0.000 (0)
   caixa_visor       0/379    0.000 (0)
   circulo             0/6    0.000 (0)
   distante_parede  16/225  0.454 (141)
   distante_poste     8/31   0.652 (23)
   grade              2/43    0.000 (0)
   parede             0/10    0.000 (0)
   raiz             1/6133  0.700 (603)
   visor             0/725    0.000 (0)
C3 caixa             1/318    0.000 (0)
   caixa_visor       0/379    0.000 (0)
   circulo             0/6    0.000 (0)
   distante_parede  15/225  0.511 (141)
   distante_poste     7/31   0.609 (23)
   grade              3/43    0.000 (0)
   parede             0/10    0.000 (0)
   raiz             1/6133  0.720 (603)
   visor             1/725    0.000 (0)

,destino trocado,duas leituras: argmax múltiplos,duas leituras: rejeitadas
C4,54/501,0/173,0
C3,57/520,0/173,0


P(rejeitável)  P(indeterminado)  P(multiplos)  P(outros)
C4 antes          0.0123            0.0122        0.0046     0.0046
   depois         0.0058            0.0057        0.0026     0.0033
C3 antes          0.0127            0.0146        0.0027     0.0055
   depois         0.0034            0.0065        0.0019     0.0028

In [16]:
# Bloco código 16
# T4g.4 — Artefato do C4, conjunto do C3 e exportação para ONNX (revisão R6)
import json

import torch

from meter_reader.config import get_work_dir
from meter_reader.triage import load_triage
from training.build_triage import ARTIFACT_NAME, C3_CALIBRATION_NAME, final_ensemble
from training.onnx_export import TRIAGE5_OUTPUT_NAMES, export_onnx
from training.run_final import FINAL_FOLDER, sha256_file

WORK_DIR = get_work_dir()
MODELS = WORK_DIR / FINAL_FOLDER

# C4: o artefato do produto, carregado pelo código do produto e conferido pelo SHA-256
c4_record = json.loads((MODELS / ARTIFACT_NAME).with_suffix(".json").read_text(encoding="utf-8"))
c4 = load_triage(MODELS / ARTIFACT_NAME, expected_sha256=c4_record["sha256_artefato"])

# C3: as 3 redes finais (conferidas pelo SHA-256) com a temperatura da execução R3
c3_record = json.loads((MODELS / C3_CALIBRATION_NAME).read_text(encoding="utf-8"))
c3 = final_ensemble(WORK_DIR, "C3", c3_record["temperatura"])
c3_threshold = c3_record["limiar"]

print(f"C4: T = {float(c4.ensemble.temperature):.4f}; limiar = {c4.threshold:.6f}; "
      f"redes = {len(c4.ensemble.nets)}")
print(f"C3: T = {float(c3.temperature):.4f}; limiar = {c3_threshold:.6f}; "
      f"redes = {len(c3.nets)}")

# Exportação: um grafo por conjunto (3 redes, média dos logits e temperatura),
# com altura e largura variáveis. Exemplos na forma de uma foto em pé.
ONNX_PATHS = {"C4": MODELS / "triagem_c4.onnx", "C3": MODELS / "triagem_c3.onnx"}
EXAMPLES = {"C4": torch.randn(1, 3, 480, 360), "C3": torch.randn(1, 3, 294, 224)}
for name, ensemble in (("C4", c4.ensemble), ("C3", c3)):
    path = ONNX_PATHS[name]
    if not path.exists():  # nunca sobrescreve uma exportação já gravada
        export_onnx(ensemble, path, EXAMPLES[name], TRIAGE5_OUTPUT_NAMES)
    print(f"{name}: ONNX de {path.stat().st_size / 2**20:.1f} MB; SHA-256 {sha256_file(path)[:12]}…")

C4: T = 1.3999; limiar = 0.847161; redes = 3
C3: T = 1.4584; limiar = 0.820447; redes = 3


W1106 22:26:01.650000 50952 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


C4: ONNX de 35.2 MB; SHA-256 e785c6d1836e…
C3: ONNX de 256.4 MB; SHA-256 d5a175769e2f…


In [17]:
# Bloco código 17
# T4g.4 — Paridade PyTorch × ONNX Runtime do artefato do C4, em CPU (revisão R6)
import time

import numpy as np

from meter_reader.preprocess import to_tensor
from meter_reader.triage import class_probs, rejection_destination, rejection_score
from training.finetune import PhotoDataset
from training.group_dataset import LABEL_VERSIONS, load_group_photos
from training.onnx_export import cpu_session, run_onnx_triage5
from training.run_finetune import eval_order

THREADS = 4  # piso da E04
photos = load_group_photos(LABEL_VERSIONS["r3"])
order = eval_order(photos)  # pareadas dos 3 lotes, na ordem de avaliação
dev_set = PhotoDataset([photos[i].path for i in order], [photos[i].pseudonym for i in order],
                       [0] * len(order), to_tensor, train=False)  # realce, sem aumentação

torch.set_num_threads(THREADS)
session = cpu_session(ONNX_PATHS["C4"], threads=THREADS)
logits_pt, logits_onnx = [], []
start = time.perf_counter()
with torch.no_grad():
    for k in range(len(dev_set)):
        x, _ = dev_set[k]
        x = x.unsqueeze(0)  # uma foto por vez, como no produto
        logits_pt.append(c4.ensemble(x).numpy()[0])
        logits_onnx.append(run_onnx_triage5(session, x.numpy())[0])
        if (k + 1) % 1000 == 0:
            print(f"{k + 1} de {len(dev_set)} fotos ({(time.perf_counter() - start) / 60:.1f} min)")
logits_pt, logits_onnx = np.stack(logits_pt), np.stack(logits_onnx)

# Mesmas contas do produto sobre as duas saídas
probs_pt = class_probs(torch.from_numpy(logits_pt))
probs_onnx = class_probs(torch.from_numpy(logits_onnx))
score_pt, score_onnx = rejection_score(probs_pt).numpy(), rejection_score(probs_onnx).numpy()
rej_pt, rej_onnx = score_pt >= c4.threshold, score_onnx >= c4.threshold
dest_pt, dest_onnx = np.array(rejection_destination(probs_pt)), np.array(rejection_destination(probs_onnx))

print(f"Fotos: {len(order)}")
print(f"Diferença máxima dos logits: {np.abs(logits_pt - logits_onnx).max():.2e}")
print(f"Diferença máxima do escore: {np.abs(score_pt - score_onnx).max():.2e}")
print(f"Folga mínima até o limiar (PyTorch): {np.abs(score_pt - c4.threshold).min():.2e}")
print(f"Decisões de rejeição divergentes: {int((rej_pt != rej_onnx).sum())}")
print(f"Destinos divergentes entre as rejeitadas: {int((dest_pt[rej_pt] != dest_onnx[rej_pt]).sum())}")
print(f"Rejeitadas pelo C4 final: {int(rej_pt.sum())} de {len(order)}")

1000 de 8637 fotos (1.6 min)
2000 de 8637 fotos (3.6 min)
3000 de 8637 fotos (5.7 min)
4000 de 8637 fotos (7.9 min)
5000 de 8637 fotos (11.0 min)
6000 de 8637 fotos (13.4 min)
7000 de 8637 fotos (15.6 min)
8000 de 8637 fotos (17.7 min)
Fotos: 8637
Diferença máxima dos logits: 1.17e-05
Diferença máxima do escore: 1.37e-06
Folga mínima até o limiar (PyTorch): 1.91e-04
Decisões de rejeição divergentes: 0
Destinos divergentes entre as rejeitadas: 0
Rejeitadas pelo C4 final: 673 de 8637
